In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import math
import numpy as np
from sklearn.model_selection import train_test_split

base = Path("C:/Users/ARNAV TRIPATHI/Desktop/ANCHOLIK-NER/datasets")

barishal = pd.read_csv(base / "Barishal_NER.csv")
chittagong = pd.read_csv(base / "Chittagong_NER.csv")
mymensingh = pd.read_csv(base / "Mymensingh_NER.csv")
noakhali = pd.read_csv(base / "Noakhali_NER.csv")
sylhet = pd.read_csv(base / "Sylhet_NER.csv")

datasets = {
    "Barishal": barishal,
    "Chittagong": chittagong,
    "Mymensingh": mymensingh,
    "Noakhali": noakhali,
    "Sylhet": sylhet
}

In [2]:
sentence_data = {}

for dialect, df in datasets.items():

    df["Sentence #"] = df["Sentence #"].ffill()

    word_column = [
        c for c in df.columns
        if c.endswith("_word")
    ][0]

    sentences = []

    for sentence_id, group in df.groupby("Sentence #"):

        sentences.append({
            "sentence_id": int(
                str(sentence_id).replace("Sentence: ", "")
            ),
            "tokens": group[word_column].astype(str).tolist(),
            "tags": group["bio_tag"].astype(str).tolist()
        })

    sentence_data[dialect] = sentences

print("Sentence conversion complete.")

Sentence conversion complete.


In [3]:
print("Barishal:", len(sentence_data["Barishal"]))
print("Chittagong:", len(sentence_data["Chittagong"]))
print("Mymensingh:", len(sentence_data["Mymensingh"]))
print("Noakhali:", len(sentence_data["Noakhali"]))
print("Sylhet:", len(sentence_data["Sylhet"]))

Barishal: 3481
Chittagong: 3481
Mymensingh: 3481
Noakhali: 3481
Sylhet: 3481


In [4]:
sentence_data["Sylhet"][0]

{'sentence_id': 1, 'tokens': ['ভালা', 'আছনি'], 'tags': ['O', 'O']}

In [5]:
for dialect, sentences in sentence_data.items():

    for sentence in sentences:

        assert len(sentence["tokens"]) == len(sentence["tags"])

print("All sentences have matching token and tag lengths.")

All sentences have matching token and tag lengths.


In [7]:
splits = {}

for dialect, sentences in sentence_data.items():

    sentence_ids = [
        s["sentence_id"]
        for s in sentences
    ]

    train_ids, test_ids = train_test_split(
        sentence_ids,
        test_size=0.20,
        random_state=42
    )

    splits[dialect] = {
        "train": sorted(train_ids),
        "test": sorted(test_ids)
    }

print("Splits created.")

Splits created.


In [8]:
for dialect, split in splits.items():

    print(
        dialect,
        "| Train:", len(split["train"]),
        "| Test:", len(split["test"])
    )

Barishal | Train: 2784 | Test: 697
Chittagong | Train: 2784 | Test: 697
Mymensingh | Train: 2784 | Test: 697
Noakhali | Train: 2784 | Test: 697
Sylhet | Train: 2784 | Test: 697


In [9]:
for dialect, split in splits.items():

    overlap = set(split["train"]) & set(split["test"])

    print(dialect, "| Overlap:", len(overlap))

Barishal | Overlap: 0
Chittagong | Overlap: 0
Mymensingh | Overlap: 0
Noakhali | Overlap: 0
Sylhet | Overlap: 0


In [10]:
import json

output = base / "ANCHOLIK_Phase2"
output.mkdir(exist_ok=True)

with open(output / "within_dialect_splits.json", "w") as f:
    json.dump(splits, f, indent=4)

print("Split IDs saved.")

Split IDs saved.
